# Ensemble on one sheet: cue_dropout + reader-v3 + v8-in (the three mix6 models)

The three mix6 models, each where its training leaves it (`utils/mix6.py` `MODEL_PATHS`):

| model | trained by | file |
|---|---|---|
| cue_dropout_labelsv6_mix6 (sequence model, lowest-validation-loss checkpoint) | `python3 train.py` | `models/runs/cue_dropout_labelsv6_mix6/best_model_loss.pth` |
| reader-v3 fine-tune, 2000 optimizer steps | `python3 finetune_readerv3.py` | `reader-v3/runs/mix6/ckpt_002000.pth` |
| v8-in fine-tune, epoch 3 of 5 | `python3 finetune_v8in.py` | `nader-fork/runs/v8in_mix6_5ep/runs/v8in_mix6_5ep_20261027_151524_4edc6c20/checkpoints/epochepoch=2.ckpt` |

The released reader-v3 (`reader-v3/reader-v3.pth`, no fine-tuning) is predicted too and shown for comparison only; it does not contribute to the ensemble.

Each model predicts the sheet at its own resolution (v8-in and reader-v3 per pixel, the sequence model per 16 px cell, repeated up to pixels, nothing is
downsampled). Every map is contrast-stretched to its own p1-p99.5 over the sheet mask; the three mix6 maps are combined two ways: `SCORING`
('z-score' = mean of the per-model z-scores) and a mid-boost composite (`utils/depth_ensemble.py`). `run_ensemble(sheet)` saves the 3x2 grid
(reader-v3 base | reader-v3 fine-tune / cue_dropout | v8-in / z-score composite | mid-boost composite) at full resolution, the two composites as plain PNGs
next to it (`ensemble_z_score.png`, `ensemble_mid_boost.png`), and shows the grid at 1/10; each sheet has its own cell at the bottom.

Define the models, the sheets and the scoring in the first code cell, then run the rest. Predictions are cached under `output/ensemble/<OUTPUT_SUBDIR>/<sheet>/cache/`
(delete a file there to recompute that model). v8-in and reader-v3 run in their own virtualenvs as subprocesses (`utils/predict_models.py`); the sequence model runs in this kernel.
v8-in reads the central 24 layers, reader-v3 the 17 central layers of its checkpoint, the sequence model a flat depth-13 surface with window offset -1.


In [ ]:
import sys
from pathlib import Path

REPO = Path("/vesuvius")
sys.path.insert(0, str(REPO))
from utils.mix6 import MODEL_PATHS

# ---- sheets: name -> render zarr / mask id, and the flat-alias id (cached normalisation, stored cue map) -----------------------
SHEETS = {
    "auto_grown_20261012065803387_straightened2_abf": {"zid": "20261012065814", "norm_id": 20261012065815},
}

# ---- the models, in figure order: name -> settings (kind: cue_dropout | reader_v3 | v8in, see utils/predict_models.detect_kind) --
# "in_ensemble": False = shown in the figure only, not part of the composites. The name is the cache file name: renaming recomputes.
MODELS = {
    # released checkpoint, no fine-tuning
    "reader-v3 base (no fine-tuning)": {"kind": "reader_v3", "checkpoint": REPO / "reader-v3" / "reader-v3.pth", "batch_size": 4,
                                        "in_ensemble": False},
    # finetune_readerv3.py, 2000 optimizer steps
    "reader-v3 (mix6 fine-tune, step 2000)": {"kind": "reader_v3", "checkpoint": MODEL_PATHS["reader_v3"], "batch_size": 4},
    # train.py; flat depth-13 surface, window offset -1
    "cue_dropout (mix6)": {"kind": "cue_dropout", "checkpoint": MODEL_PATHS["cue_dropout"], "depth_offset": -1, "infer_batch": 16},
    # finetune_v8in.py, epoch 3 Lightning checkpoint (MODEL_PATHS["v8in"] is the exported final model)
    "v8-in (mix6 fine-tune, epoch 3)": {"kind": "v8in", "batch_size": 16, "checkpoint": REPO / "nader-fork" / "runs" / "v8in_mix6_5ep" / "runs"
                                        / "v8in_mix6_5ep_20261027_151524_4edc6c20" / "checkpoints" / "epochepoch=2.ckpt"},
}

# ---- scoring ----------------------------------------------------------------------------------------------------------
SCORING = "z-score"      # 'z-score' | 'mean' | 'rank'  (see SCORING_METHODS below)
STRETCH = (1.0, 99.5)    # percentiles of each model's map over the sheet mask

FLAT_DEPTH = 13
EVAL_STRIDE = 32         # cue_dropout inference stride (px)
PREVIEW_DOWNSCALE = 10   # the saved figure stays full size
GRID_COLS = 2            # panels per row: reader base | reader fine-tune / cue_dropout | v8-in / z-score composite | mid-boost composite
OUTPUT_SUBDIR = "mix6"   # output/ensemble/<OUTPUT_SUBDIR>/<sheet>/ ; keeps these runs apart from the earlier ensembles


In [ ]:
import gc
import os
import types

import cv2
import numpy as np
import torch
import zarr
from IPython.display import display
from PIL import Image

# nonml_ink.py (imported by utils.precomputed_asf) lives in old/
for _path in (REPO, REPO / "old"):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))
os.chdir(REPO)
Image.MAX_IMAGE_PIXELS = None

from utils import predict_models
from utils.mix6 import zarr_mtime
from utils.fast_figure import colorize, orient_display, save_panel_grid

ENSEMBLE_ROOT = REPO / "output" / "ensemble" / OUTPUT_SUBDIR


def slug(text):
    return "".join(ch if ch.isalnum() else "_" for ch in text).strip("_")


class Sheet:
    def __init__(self, name):
        spec = SHEETS[name]
        self.name, self.zid, self.norm_id = name, spec["zid"], spec["norm_id"]
        self.zarr_path = REPO / "ves_zarrs2" / f"{self.zid}.zarr"
        self.out_dir = ENSEMBLE_ROOT / name
        self.cache = self.out_dir / "cache"
        self.cache.mkdir(parents=True, exist_ok=True)
        self.mask = cv2.imread(str(REPO / "masks" / f"{self.zid}.png"), cv2.IMREAD_GRAYSCALE) > 0
        self.frame = self.mask.shape

    def to_frame(self, array):
        # map of any extent -> float32 frame, NaN outside the sheet mask
        out = np.full(self.frame, np.nan, np.float32)
        rows, cols = min(self.frame[0], array.shape[0]), min(self.frame[1], array.shape[1])
        out[:rows, :cols] = array[:rows, :cols]
        out[~self.mask] = np.nan
        return out

In [ ]:
# ---- v8-in and reader-v3: subprocesses in their own virtualenvs (utils/predict_models.py) ------------------------------------
def run_v8in(sheet, name, spec):
    # the central 24 layers of the render are staged in a scratch folder (removed afterwards) when the prediction is not cached
    return predict_models.predict_v8in(sheet.zarr_path, spec["checkpoint"], sheet.cache / slug(name),
                                       batch_size=spec.get("batch_size", 16))


def run_reader_v3(sheet, name, spec):
    return predict_models.predict_reader(sheet.zarr_path, spec["checkpoint"], sheet.cache / f"{slug(name)}.tif",
                                         batch_size=spec.get("batch_size", 4))


# ---- cue_dropout: in this kernel, replaying the checkpoint's run config as 0846A_eval.ipynb does ----------------------------
def run_cue_dropout(sheet, name, spec):
    path = sheet.cache / f"{slug(name)}.npz"
    # a regenerated sheet or a retrained checkpoint is newer than the cached prediction
    if not path.is_file() or path.stat().st_mtime < max(zarr_mtime(sheet.zarr_path), Path(spec["checkpoint"]).stat().st_mtime):
        from utils.config import Config, ScrollConfig, resolve_run_config
        from utils.model import create_model
        from utils.norm import UNIFIED_CACHE_PATH, load_cached_norm
        from utils.platform import get_zarr_dir
        from utils.visualizer import TensorboardVisualizer, predict_tiles

        checkpoint = Path(spec["checkpoint"]).resolve()
        run_config, _ = resolve_run_config(checkpoint)
        C = Config()
        for section in ("data", "model"):
            for key, value in run_config[section].items():
                if section == "data" and key == "scrolls":
                    C.data.scrolls = [ScrollConfig(**item) for item in value]
                elif key != "probe_rois" and hasattr(getattr(C, section), key):
                    setattr(getattr(C, section), key, value)
        C.model.compile_model = False
        C.tra.supcon = False
        C.tra.dann = False
        C.device = "cuda" if torch.cuda.is_available() else "cpu"
        C.data.eval_stride = EVAL_STRIDE
        C.data.eval_infer_bs = int(spec.get("infer_batch", 0))  # 0 = automatic; a small value saves VRAM, results are identical
        offset = int(spec.get("depth_offset", -1))
        C.data.surface_window_offset = offset
        C.data.surface_window_offset_by_scroll = {}
        C.data.zarr_path = str(Path(get_zarr_dir()))
        depth = int(C.data.depth)
        if depth != 8 or not C.model.surface_teacher_input or not C.data.surface_relative_depth_window:
            raise ValueError("Select a literal-surface, eight-slice checkpoint")
        model, _ = create_model(C)
        state = torch.load(checkpoint, map_location="cpu", weights_only=True)
        state = state.get("model_state_dict", state.get("state_dict", state))
        state = {key.removeprefix("module.").removeprefix("_orig_mod."): value for key, value in state.items()}
        state = {key: value for key, value in state.items() if not key.startswith(("supcon_head.", "domain_head."))}
        model.load_state_dict(state, strict=True)
        model.eval()

        volume = predict_models.read_volume(predict_models.local_zarr(sheet.zarr_path))
        volume.setflags(write=False)
        height, width = sheet.mask.shape
        surface_depth = np.where(sheet.mask, FLAT_DEPTH, 255).astype(np.uint8)
        surface_confidence = np.where(sheet.mask, 255, 0).astype(np.uint8)
        norm = load_cached_norm(sheet.norm_id, UNIFIED_CACHE_PATH, mode=str(C.data.norm_mode))
        if norm is None:
            raise RuntimeError(f"no cached normalisation for {sheet.norm_id}")
        cue = None
        if str(C.model.input_cue) == "pre_clahe":
            cue = np.load(Path(C.data.cue_dir) / f"{sheet.norm_id}.npy", mmap_mode="r")
        coords = TensorboardVisualizer._gen_tile_coords(
            types.SimpleNamespace(c=C), (C.data.d_start, C.data.d_end), (0, height), (0, width), sheet.mask, z_step=depth,
        )
        grid = predict_tiles(
            C, model, volume, sheet.mask, coords, (0, height), (0, width), C.data.d_start, sheet.name, *norm,
            surface_depth_map=surface_depth, surface_confidence_map=surface_confidence,
            surface_depth_offset=offset, cue_map=cue,
        )
        np.savez(path, grid=np.asarray(grid, dtype=np.float32), cell=int(C.model.multitile_subtile))
        del model, volume
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    saved = np.load(path)
    cell = int(saved["cell"])
    return np.repeat(np.repeat(saved["grid"], cell, axis=0), cell, axis=1)


RUNNERS = {"cue_dropout": run_cue_dropout, "reader_v3": run_reader_v3, "v8in": run_v8in}

In [ ]:
# ---- stretching and scoring ----------------------------------------------------------------------------------------------
from utils import depth_ensemble


def stretch(prediction, mask):
    valid = mask & np.isfinite(prediction)
    low, high = np.percentile(prediction[valid], STRETCH)
    stretched = np.clip((prediction - low) / max(float(high - low), 1e-6), 0, 1)
    return np.where(valid, stretched, np.nan).astype(np.float32), float(low), float(high)


def z_score(values):
    valid = np.isfinite(values)
    return ((values - values[valid].mean()) / max(float(values[valid].std()), 1e-6)).astype(np.float32)


def rank_score(values):
    valid = np.isfinite(values)
    ranks = np.full(values.shape, np.nan, np.float32)
    order = np.argsort(values[valid], kind="stable")
    scores = np.empty(order.size, np.float32)
    scores[order] = np.linspace(0, 1, order.size, dtype=np.float32)
    ranks[valid] = scores
    return ranks


# stretched maps (list) -> ensemble map; every method returns NaN outside the sheet
SCORING_METHODS = {
    "z-score": lambda maps: np.mean([z_score(m) for m in maps], axis=0),
    "mean": lambda maps: np.mean(maps, axis=0),
    "rank": lambda maps: np.mean([rank_score(m) for m in maps], axis=0),
}
if SCORING not in SCORING_METHODS:
    raise ValueError(f"SCORING must be one of {sorted(SCORING_METHODS)}")

RED = (220, 0, 0)


def run_ensemble(sheet_name):
    # run every model on the sheet (cached predictions are reused), stretch, combine the models with in_ensemble != False
    # (z-score composite and mid-boost composite), save the 3x2 grid at full resolution plus both composites as plain PNGs
    sheet = Sheet(sheet_name)
    print(f"{sheet.name}: frame {sheet.frame}, {int(sheet.mask.sum()):,} papyrus pixels; cache {sheet.cache}")
    stretched, limits = {}, {}
    for name, spec in MODELS.items():
        print(f"[{spec['kind']}] {name}: {spec['checkpoint']}")
        if predict_models.detect_kind(spec["checkpoint"]) != spec["kind"]:
            raise ValueError(f"{spec['checkpoint']} is not a {spec['kind']} model")
        stretched[name], low, high = stretch(sheet.to_frame(RUNNERS[spec["kind"]](sheet, name, spec)), sheet.mask)
        limits[name] = (low, high)
        print(f"    p{STRETCH[0]:g} {low:.4f}, p{STRETCH[1]:g} {high:.4f}")
        gc.collect()

    members = [name for name, spec in MODELS.items() if spec.get("in_ensemble", True)]
    maps = [stretched[name] for name in members]
    composites = {
        f"{SCORING}": SCORING_METHODS[SCORING](maps),
        "mid-boost": depth_ensemble.combine(np.stack(maps), "mid-boost"),
    }
    bad = (0.78, 0.86, 0.95)
    composite_rgb = {}
    for label, array in composites.items():
        valid = np.isfinite(array)
        c_low, c_high = np.percentile(array[valid], STRETCH)
        print(f"{label} composite of {len(members)}: p{STRETCH[0]:g} {c_low:.3f}, p{STRETCH[1]:g} {c_high:.3f}")
        composite_rgb[label] = colorize(orient_display(array, sheet.name), "gray_r", bad, vmin=float(c_low), vmax=float(c_high))
        np.save(sheet.out_dir / f"ensemble_{slug(label)}.npy", array.astype(np.float16))
        png = sheet.out_dir / f"ensemble_{slug(label)}.png"
        cv2.imwrite(str(png), composite_rgb[label][..., ::-1], [cv2.IMWRITE_PNG_COMPRESSION, 1])
        print(f"Saved {png}")

    panels = []
    for name, array in stretched.items():
        title = f"{name} | stretched p{STRETCH[0]:g}-p{STRETCH[1]:g} ({limits[name][0]:.3f}..{limits[name][1]:.3f})"
        if not MODELS[name].get("in_ensemble", True):
            title = (f"{title} | DOES NOT CONTRIBUTE TO THE ENSEMBLE", RED)
        panels.append((title, colorize(orient_display(array, sheet.name), "gray_r", bad), cv2.INTER_AREA, None))
    panels.append((f"{SCORING} composite of {len(members)} models | mean of per-model {SCORING}s, shown p{STRETCH[0]:g}-p{STRETCH[1]:g}",
                   composite_rgb[SCORING], cv2.INTER_AREA, None))
    panels.append((f"mid-boost composite of {len(members)} models | shown p{STRETCH[0]:g}-p{STRETCH[1]:g}",
                   composite_rgb["mid-boost"], cv2.INTER_AREA, None))
    cell_hw = composite_rgb[SCORING].shape[:2]

    rows = [panels[i:i + GRID_COLS] for i in range(0, len(panels), GRID_COLS)]
    grid_path = sheet.out_dir / f"ensemble_{slug(SCORING)}_{len(rows)}x{GRID_COLS}.png"
    save_panel_grid(rows, cell_hw, str(grid_path), gap_px=24)
    print(f"Saved {grid_path}")
    with Image.open(grid_path) as saved:
        print("full size", saved.size)
        display(saved.resize((max(1, saved.width // PREVIEW_DOWNSCALE), max(1, saved.height // PREVIEW_DOWNSCALE)),
                             Image.Resampling.LANCZOS))
    del stretched, composites, composite_rgb, panels
    gc.collect()
    return grid_path


In [ ]:
# straightened2 (held-out winning sheet)
straightened2_path = run_ensemble("auto_grown_20261012065803387_straightened2_abf")